# XGBoost Donor-Request Matching & Handover Predictor
This notebook trains an XGBoost classifier to predict the probability that a donor will complete a donation handover to a disaster officer.

### Target Definition
- `is_completed = 1`: Donation items successfully received and handed over to disaster officer (`status == 'received'`).
- `is_completed = 0`: Donation was cancelled, expired, or unfulfilled.

## Step 1: Install Required Packages

In [ ]:
!pip install xgboost scikit-learn pandas numpy joblib matplotlib seaborn

## Step 2: Upload Dataset (`donor_matching_training_data.csv`)
Run this cell and select `donor_matching_training_data.csv` from your local machine, or upload it via the left Files panel.

In [ ]:
import os
from google.colab import files

if not os.path.exists('donor_matching_training_data.csv'):
    print('Please upload donor_matching_training_data.csv:')
    uploaded = files.upload()
else:
    print('donor_matching_training_data.csv is already uploaded!')

## Step 3: Load and Inspect the Data

In [ ]:
import pandas as pd

df = pd.read_csv('donor_matching_training_data.csv')
print(f"Total Samples: {len(df):,}")
print(f"Target Distribution:\n{df['is_completed'].value_counts(normalize=True).round(3) * 100}")
df.head()

## Step 4: Build Preprocessing Pipeline & Train XGBoost

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
)
import joblib

categorical_features = [
    "donor_type",
    "preferred_category",
    "capacity_level",
    "disaster_type",
    "severity",
    "required_category",
]

numeric_features = [
    "affected_population",
    "required_quantity",
    "distance_km",
    "category_match_score",
    "capacity_match_score",
    "proximity_score",
    "past_fulfillment_rate",
    "days_since_last_donation",
]

feature_cols = categorical_features + numeric_features
target_col = "is_completed"

X = df[feature_cols]
y = df[target_col]

# 80/20 Stratified Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("num", "passthrough", numeric_features),
    ]
)

# XGBoost Classifier
classifier = xgb.XGBClassifier(
    n_estimators=180,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    eval_metric="logloss",
    random_state=42,
)

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", classifier),
])

print("Training XGBoost Model...")
model_pipeline.fit(X_train, y_train)
print("Done!")

## Step 5: Evaluate Performance

In [ ]:
y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("=" * 50)
print("MODEL EVALUATION METRICS:")
print("=" * 50)
print(f"  Accuracy:   {acc:.4f} ({acc*100:.2f}%)")
print(f"  Precision:  {prec:.4f}")
print(f"  Recall:     {rec:.4f}")
print(f"  F1 Score:   {f1:.4f}")
print(f"  ROC-AUC:    {auc:.4f}")
print("-" * 50)
print(classification_report(y_test, y_pred, target_names=['Unfulfilled (0)', 'Handover Completed (1)']))

## Step 6: Feature Importances

In [ ]:
import matplotlib.pyplot as plt

ohe = model_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
cat_names = list(ohe.get_feature_names_out(categorical_features))
all_feature_names = cat_names + numeric_features
importances = model_pipeline.named_steps["classifier"].feature_importances_

feat_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feat_df['Feature'].head(12)[::-1], feat_df['Importance'].head(12)[::-1], color='#059669')
plt.xlabel('Importance Score')
plt.title('Top 12 Features for Donor Handover Prediction')
plt.tight_layout()
plt.show()

## Step 7: Export and Download Model Package (`donor_xgboost_model.joblib`)

In [ ]:
model_package = {
    "model": model_pipeline,
    "model_name": "XGBoost Donor Matching & Handover Predictor",
    "algorithm": "XGBClassifier (Gradient Boosted Decision Trees)",
    "version": "1.0.0",
    "feature_columns": feature_cols,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "metrics": {
        "accuracy": round(float(acc), 4),
        "precision": round(float(prec), 4),
        "recall": round(float(rec), 4),
        "f1": round(float(f1), 4),
        "roc_auc": round(float(auc), 4),
    },
    "target_description": "is_completed (Handover completed to disaster officer)",
}

joblib.dump(model_package, "donor_xgboost_model.joblib")
print("Model saved as 'donor_xgboost_model.joblib'!")

# Trigger automatic browser download in Colab
from google.colab import files
files.download("donor_xgboost_model.joblib")